# 第 07 课：自动检查与基础调试

**前置知识：** 列表、字典、循环、函数、异常处理，以及上一课的 main 与程序入口。  
**建议用时：** 2–3 小时，另留时间独立完成练习。  
**配套脚本：** `lesson_07_testing_and_debugging.py`。

今天从“程序打印了一些结果”前进一步：先写清正确结果，再让代码自动比较；出现差异时，用一个小输入复现、定位、修复，并重新检查原有行为。

本课继续使用固定字符词表，不需要模型、GPU 或测试框架。编码、解码函数都在文件中完整提供，不依赖上一课的文件。

完整脚本包含 3 个测试函数、20 项固定检查。讲义还提供故障示例和变化练习；这些额外演示不计入脚本的 20 项。


## 使用方法

在普通 Python 3 内核中从上到下运行。分步示例会使用前面的定义；“完整程序”单元与配套 .py 源码一致，可以独立运行。三个参考答案各自带齐依赖，也可以独立复制运行。

讲义中故意失败的例子会标注为“故障演示”，并捕获预期的 AssertionError，让后面的单元继续执行。最终完整程序不会把测试失败吞掉：失败会停止执行，只有所有检查通过才打印总结果。

本课程序只进行计算和打印，不生成或修改数据文件。配套 .py 被导入时不会自动运行测试。

练习起始单元保留了合法代码或注释，因此“运行全部”可以完成；学习验收仍需独立写出练习。修改某个函数后，要重新运行该函数定义和相应测试。完整程序单元会重新定义标准版 encode/decode；故障版另用 encode_buggy 命名。

使用普通 Python 启动方式，不加 -O 或 -OO。它们可能移除 assert；本课入口也会对此进行检查。


## 1. 准备已经熟悉的函数

以下 encode/decode 与第 06 课相同。已知字符“我、爱、学、习、模、型”对应 0–5，未知标记 <unk> 对应 6。

输入检查仍使用 if 与 raise ValueError。词表本身由示例保证有效；本课的测试以这份固定词表为约定。


In [6]:
if not __debug__:
    raise RuntimeError("请使用未启用 -O / -OO 的普通 Python 内核运行本课。")

def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored


token_to_id = {
    "我": 0, "爱": 1, "学": 2, "习": 3,
    "模": 4, "型": 5, "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]


## 2. assert：把预期写成可以执行的检查

基本写法是：

```python
assert 条件, "条件不成立时的说明"
```

条件成立时继续往下运行；条件不成立时抛出 AssertionError。assert 是语句，直接按这个形式写，不要把“条件和说明”一起包进一对括号。

先从词表手算“我爱”应得到 [0, 1]，再运行函数。预期值应来自明确的规则，不应再次调用被测函数来计算。


In [7]:
expected = [0, 1]
actual = encode("我爱", token_to_id)
assert actual == expected, "我爱 应编码为 [0, 1]"
print("通过：", actual)


通过： [0, 1]


接下来故意给出一个错误的实际结果，观察失败。这里只捕获教学演示的错误，不表示这个检查通过了。


In [8]:
try:
    actual = [0]
    expected = [0, 1]
    assert actual == expected, "结果少了第二个编号"
except AssertionError as error:
    print("已捕获故障演示：", error)


已捕获故障演示： 结果少了第二个编号


## 3. 让失败信息包含输入、预期和实际结果

`repr(值)` 返回便于观察的字符串表示。它能把空字符串显示为引号，让你区分空字符串、空列表和 None，也能显示文字中的空格。

比较列表时，本课关心内容与顺序。仅检查 len(actual) 不够：[1, 0] 与 [0, 1] 长度相同，含义却不同。


In [9]:
print("空字符串：", repr(""))
print("空列表：", repr([]))
print("没有返回值：", repr(None))
print("带空格的文字：", repr("我 爱"))

actual = [0]
expected = [0, 1]
message = "输入：" + repr("我爱")
message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
print(message)


空字符串： ''
空列表： []
没有返回值： None
带空格的文字： '我 爱'
输入：'我爱'，预期：[0, 1]，实际：[0]


## 4. 把多个固定案例组织起来

一条测试案例包含输入和对应预期。下面用已经学过的“列表里放字典”组织案例，再用 for 循环逐项检查。

| 类别 | 输入 | 固定预期 |
| --- | --- | --- |
| 已知字符 | 我爱学习 | [0, 1, 2, 3] |
| 已知与未知混合 | 我爱大模型 | [0, 1, 6, 4, 5] |
| 空字符串 | 空字符串 | [] |
| 重复已知字符 | 我我爱 | [0, 0, 1] |
| 重复未知字符 | 大大 | [6, 6] |

后面再连续调用两次 encode，检查结果是否相互独立。五条输入案例加两条连续调用检查，共 7 项。

test_encode_cases 是普通函数。**test_ 前缀只是这里的命名习惯，没有测试框架时，必须自己调用。** 函数全部执行完才返回检查数量；任意断言失败都会提前中断。


In [10]:
def test_encode_cases(token_to_id):
    cases = [
        {"name": "已知字符", "text": "我爱学习", "expected": [0, 1, 2, 3]},
        {"name": "混合未知字符", "text": "我爱大模型", "expected": [0, 1, 6, 4, 5]},
        {"name": "空字符串", "text": "", "expected": []},
        {"name": "重复已知字符", "text": "我我爱", "expected": [0, 0, 1]},
        {"name": "重复未知字符", "text": "大大", "expected": [6, 6]},
    ]
    for case in cases:
        actual = encode(case["text"], token_to_id)
        expected = case["expected"]
        message = case["name"] + "，输入：" + repr(case["text"])
        message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
        assert actual == expected, message

    first = encode("我爱", token_to_id)
    second = encode("学习", token_to_id)
    assert first == [0, 1], "连续调用后第一次结果不应变化，实际：" + repr(first)
    assert second == [2, 3], "第二次调用不应累积旧编号，实际：" + repr(second)
    return len(cases) + 2


print("编码检查通过项数：", test_encode_cases(token_to_id))


编码检查通过项数： 7


## 5. 解码也要有独立的预期

不能只检查 decode(encode(text)) 是否等于 text。如果两个函数中的错误恰好抵消，往返检查仍可能通过。这里为解码直接写出编号列表与预期文字。

本课允许编号 6，解码结果是字符串 <unk>。它不能恢复原来的未知字符，因此不要要求含未知字符的文本一定能原样往返。


In [11]:
def test_decode_cases(id_to_token):
    cases = [
        {"name": "已知编号", "token_ids": [0, 1, 2, 3], "expected": "我爱学习"},
        {"name": "未知标记", "token_ids": [6], "expected": "<unk>"},
        {"name": "空编号列表", "token_ids": [], "expected": ""},
        {"name": "重复编号", "token_ids": [0, 0, 1], "expected": "我我爱"},
    ]
    for case in cases:
        actual = decode(case["token_ids"], id_to_token)
        expected = case["expected"]
        message = case["name"] + "，输入：" + repr(case["token_ids"])
        message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
        assert actual == expected, message
    return len(cases)


print("解码检查通过项数：", test_decode_cases(id_to_token))


解码检查通过项数： 4


## 6. 错误输入也有预期：应该抛出什么异常？

对错误输入，不能只要求“没有崩溃”。例如 decode([-1], ...) 应主动抛出 ValueError，拒绝使用负编号。

下面用一个布尔标志记录是否捕获了预期异常：每次检查前设为 False，捕获 ValueError 才改为 True，最后用 assert 检查。

没有抛异常时，标志仍为 False，测试会失败；如果抛出其他类型的异常，本段不会把它当成正确结果，而会让它继续暴露。


In [12]:
raised_expected_error = False
try:
    decode([-1], id_to_token)
except ValueError as error:
    raised_expected_error = True
    print("捕获到预期的 ValueError：", error)

assert raised_expected_error, "decode 应拒绝负编号"
print("通过：负编号确实被拒绝")


捕获到预期的 ValueError： 编号不能小于 0
通过：负编号确实被拒绝


扩展到多条错误输入时，标志要在**每次循环内部**重新设为 False，不能让上一条案例的成功掩盖下一条的失败。也不要在 except 中直接 return，否则后面的案例就没有检查。

下方检查 3 个非字符串编码输入，以及 6 个不合法的解码输入。这里的 7 是越界编号，True 是布尔值，"1" 是字符串，1.0 是浮点数。


In [13]:
def test_invalid_inputs(token_to_id, id_to_token):
    bad_texts = [123, None, []]
    for bad_text in bad_texts:
        raised_expected_error = False
        try:
            encode(bad_text, token_to_id)
        except ValueError:
            raised_expected_error = True
        assert raised_expected_error, "encode 应拒绝这个输入：" + repr(bad_text)

    bad_token_lists = ["0,1", [-1], [7], [True], ["1"], [1.0]]
    for bad_ids in bad_token_lists:
        raised_expected_error = False
        try:
            decode(bad_ids, id_to_token)
        except ValueError:
            raised_expected_error = True
        assert raised_expected_error, "decode 应拒绝这个输入：" + repr(bad_ids)

    return len(bad_texts) + len(bad_token_lists)


print("错误输入检查通过项数：", test_invalid_inputs(token_to_id, id_to_token))


错误输入检查通过项数： 9


## 7. 一个会骗过单字符测试的错误

下面的 encode_buggy 故意把 return 缩进到 for 循环内部。先只用单字符“我”测试，看看会发生什么。

注意这里用不同函数名保存故障版，不覆盖上面的正式 encode。


In [14]:
def encode_buggy(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
        return token_ids  # 故意放在循环里，用于调试练习。


actual = encode_buggy("我", token_to_id)
assert actual == [0], "单字符检查"
print("单字符检查通过：", actual)


单字符检查通过： [0]


单字符通过了，但这只覆盖了一次循环。增加“我爱”和空字符串，预期分别是 [0, 1] 与 []。

**先预测：** 故障版实际返回什么？空输入会执行到循环里的 return 吗？


In [15]:
bug_cases = [
    {"text": "我爱", "expected": [0, 1]},
    {"text": "", "expected": []},
]
for case in bug_cases:
    actual = encode_buggy(case["text"], token_to_id)
    expected = case["expected"]
    message = "输入：" + repr(case["text"])
    message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
    try:
        assert actual == expected, message
    except AssertionError as error:
        print("已复现故障：", error)


已复现故障： 输入：'我爱'，预期：[0, 1]，实际：[0]
已复现故障： 输入：''，预期：[]，实际：None


多字符输入得到 [0]，因为第一次循环就离开了函数。空输入不会进入循环，函数走到结尾，没有执行 return，因此得到 None。

现在我们有了能稳定复现问题的小输入。不要把预期改成 [0] 或 None 来让检查通过；先回到函数约定，确认应该处理全部字符，空输入应返回空列表。


## 8. 阅读错误回溯：它告诉你哪里发现了差异

下面用标准库 traceback 把**已经捕获**的失败回溯打印出来，观察调用位置。format_exc() 只用于显示当前捕获的错误信息，不会修复程序。

阅读顺序：先看最后一行的异常类型与说明，再向上找自己的检查函数和调用位置。断言行是发现差异的位置；真正造成差异的错误可能在被调用的函数内部。


In [16]:
import traceback

def check_buggy_two_chars(token_to_id):
    actual = encode_buggy("我爱", token_to_id)
    assert actual == [0, 1], "我爱：预期 [0, 1]，实际 " + repr(actual)

try:
    check_buggy_two_chars(token_to_id)
except AssertionError:
    print("下面是故意触发并已捕获的失败回溯：")
    print(traceback.format_exc())


下面是故意触发并已捕获的失败回溯：
Traceback (most recent call last):
  File "C:\Users\24492\AppData\Local\Temp\ipykernel_43400\4263902157.py", line 8, in <module>
    check_buggy_two_chars(token_to_id)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\24492\AppData\Local\Temp\ipykernel_43400\4263902157.py", line 5, in check_buggy_two_chars
    assert actual == [0, 1], "我爱：预期 [0, 1]，实际 " + repr(actual)
           ^^^^^^^^^^^^^^^^
AssertionError: 我爱：预期 [0, 1]，实际 [0]



## 9. 在关键位置打印中间状态

只观察三个位置：当前处理哪个字符、追加后列表是什么、什么时候即将执行 return。下面的追踪函数同样故意保留错误缩进。

你会看到“我爱”只处理了“我”，就到了 return；空字符串没有进入循环。定位后应把 return 移到循环外，再删除这些临时追踪输出。


In [17]:
def encode_with_trace(text, token_to_id):
    token_ids = []
    print("函数开始，输入：", repr(text))
    for char in text:
        print("正在处理字符：", repr(char))
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
        print("追加后的列表：", token_ids)
        print("马上执行 return")
        return token_ids
    print("循环结束，但这个故障版没有执行 return")


traced_result = encode_with_trace("我爱", token_to_id)
print("最终返回：", repr(traced_result))
print()
empty_result = encode_with_trace("", token_to_id)
print("空输入返回：", repr(empty_result))


函数开始，输入： '我爱'
正在处理字符： '我'
追加后的列表： [0]
马上执行 return
最终返回： [0]

函数开始，输入： ''
循环结束，但这个故障版没有执行 return
空输入返回： None


## 10. 修复后重跑原来的失败样例

修正后的 return 与 for 对齐。循环完成后统一返回列表；空输入时，循环虽然没有执行，外面的 return 仍会返回初始化好的空列表。

除了重新检查原来的两个失败输入，还加入单字符、未知字符和重复字符，确认修改没有破坏这些行为。修复后再次运行已有检查，称为回归检查。


In [18]:
def encode_fixed(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


regression_cases = [
    {"text": "我", "expected": [0]},
    {"text": "我爱", "expected": [0, 1]},
    {"text": "", "expected": []},
    {"text": "我爱大模型", "expected": [0, 1, 6, 4, 5]},
    {"text": "我我爱", "expected": [0, 0, 1]},
]
for case in regression_cases:
    actual = encode_fixed(case["text"], token_to_id)
    assert actual == case["expected"], "修复后仍不符合预期：" + repr(case["text"])
print("修复后的回归检查通过：", len(regression_cases), "项")


修复后的回归检查通过： 5 项


## 11. 完整程序：正常运行全部检查

下一单元与配套 .py 完全一致。只有所有测试函数都正常返回，main 才会打印“全部通过”。一个函数失败后，后面的组不会继续执行；修复后再重跑。

脚本直接运行时检查 7 项编码行为、4 项解码行为、9 项错误输入行为，共 20 项；它不包含前面的故障版和临时追踪输出。

`__debug__` 是 Python 的内置常量，正常运行时为 True。启用 -O / -OO 会让 assert 被移除，所以 main 先拒绝这种模式，避免出现“检查没执行却显示通过”。日常照下面的普通命令运行即可：

```bash
python lesson_07_testing_and_debugging.py
```

assert 用于这里的开发检查；文件、用户输入和工具参数的校验继续使用 if + raise ValueError，不能依赖可能被关闭的断言。


In [19]:
# 第 07 课：自动检查与基础调试。
# 运行方法：python lesson_07_testing_and_debugging.py
# 仅使用 Python 内置功能；不依赖其他课程文件或测试框架。
# 直接运行会执行 20 项固定检查；测试失败时抛出异常，不打印全部通过。
# 导入本文件只定义函数，不自动运行测试；本程序不读写数据文件。
# assert 用于这些开发检查；外部输入检查仍由 if + raise ValueError 完成。


def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored


def test_encode_cases(token_to_id):
    cases = [
        {"name": "已知字符", "text": "我爱学习", "expected": [0, 1, 2, 3]},
        {"name": "混合未知字符", "text": "我爱大模型", "expected": [0, 1, 6, 4, 5]},
        {"name": "空字符串", "text": "", "expected": []},
        {"name": "重复已知字符", "text": "我我爱", "expected": [0, 0, 1]},
        {"name": "重复未知字符", "text": "大大", "expected": [6, 6]},
    ]
    for case in cases:
        actual = encode(case["text"], token_to_id)
        expected = case["expected"]
        message = case["name"] + "，输入：" + repr(case["text"])
        message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
        assert actual == expected, message

    first = encode("我爱", token_to_id)
    second = encode("学习", token_to_id)
    assert first == [0, 1], "连续调用后第一次结果不应变化，实际：" + repr(first)
    assert second == [2, 3], "第二次调用不应累积旧编号，实际：" + repr(second)
    return len(cases) + 2


def test_decode_cases(id_to_token):
    cases = [
        {"name": "已知编号", "token_ids": [0, 1, 2, 3], "expected": "我爱学习"},
        {"name": "未知标记", "token_ids": [6], "expected": "<unk>"},
        {"name": "空编号列表", "token_ids": [], "expected": ""},
        {"name": "重复编号", "token_ids": [0, 0, 1], "expected": "我我爱"},
    ]
    for case in cases:
        actual = decode(case["token_ids"], id_to_token)
        expected = case["expected"]
        message = case["name"] + "，输入：" + repr(case["token_ids"])
        message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
        assert actual == expected, message
    return len(cases)


def test_invalid_inputs(token_to_id, id_to_token):
    bad_texts = [123, None, []]
    for bad_text in bad_texts:
        raised_expected_error = False
        try:
            encode(bad_text, token_to_id)
        except ValueError:
            raised_expected_error = True
        assert raised_expected_error, "encode 应拒绝这个输入：" + repr(bad_text)

    bad_token_lists = ["0,1", [-1], [7], [True], ["1"], [1.0]]
    for bad_ids in bad_token_lists:
        raised_expected_error = False
        try:
            decode(bad_ids, id_to_token)
        except ValueError:
            raised_expected_error = True
        assert raised_expected_error, "decode 应拒绝这个输入：" + repr(bad_ids)

    return len(bad_texts) + len(bad_token_lists)


def main():
    # -O / -OO 会移除 assert；本课入口明确拒绝这种启动方式。
    if not __debug__:
        raise RuntimeError("本课使用 assert，请使用普通 Python 运行，不要启用 -O 或 -OO。")
    token_to_id = {
        "我": 0, "爱": 1, "学": 2, "习": 3,
        "模": 4, "型": 5, "<unk>": 6,
    }
    id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

    encode_count = test_encode_cases(token_to_id)
    print("通过：编码与连续调用", encode_count, "项")
    decode_count = test_decode_cases(id_to_token)
    print("通过：解码", decode_count, "项")
    invalid_count = test_invalid_inputs(token_to_id, id_to_token)
    print("通过：错误输入", invalid_count, "项")

    total = encode_count + decode_count + invalid_count
    print("全部通过：", total, "项固定检查。")
    return total


if __name__ == "__main__":
    main()


通过：编码与连续调用 7 项
通过：解码 4 项
通过：错误输入 9 项
全部通过： 20 项固定检查。


### 怎样理解“20 项通过”

它表示这 20 项检查在本次运行中符合预期，并不证明所有输入都正确，也不代表已经覆盖所有类型、所有词表或所有修改。

测试用例要跟着明确的约定扩展。遇到一个新错误，先留下能重现它的输入和预期，再修复实现，以后就能用这个案例检查是否复发。

对未来的模型工程，也需要先说明自己在检查什么：函数输出、张量形状、数据格式、模型任务正确率是不同层次的问题。本课先把确定性的函数检查做好。


## 12. 练习 A：补充三条编码检查

为下面三种输入手写预期编号，再写 test_more_encode_cases(token_to_id)，逐条调用 encode 并用 assert 检查。

- “模型”：全部为已知字符。
- “火星”：两个字符都未知。
- “我 爱”：中间包含一个空格；本课不会自动删除空格。

先根据词表写预期，不能用 encode 的返回值填入 expected。失败说明中包含输入、预期和实际结果。


In [ ]:
exercise_a_texts = ["模型", "火星", "我 爱"]
# 在这里手写预期值，再实现 test_more_encode_cases。
# 定义完成后，记得主动调用这个测试函数。


## 13. 练习 B：修复未知字符计数错误

下方故障版对一个未知字符可能给出正确结果。请先为以下输入写固定预期：

| 输入 | 应有的未知字符数 |
| --- | --- |
| 我爱学习 | 0 |
| 我爱大模型 | 1 |
| 我爱大大模型 | 2 |
| 空字符串 | 0 |
| 大大 | 2 |

先用包含两个未知字符的案例复现失败，再解释问题在哪一行。修复成 count_unknown，重跑上述五项检查。修改函数，不要降低预期。


In [ ]:
def count_unknown_buggy(text, token_to_id):
    token_ids = encode(text, token_to_id)
    count = 0
    for token_id in token_ids:
        if token_id == token_to_id["<unk>"]:
            count = 1  # 故意写错：请根据测试定位。
    return count


print("只有一个未知字符时：", count_unknown_buggy("我爱大模型", token_to_id))
# 在这里添加能复现问题的断言；演示失败时可以按前文捕获 AssertionError。
# 修复函数后，用五个案例做回归检查。


## 14. 练习 C：补充异常检查，并保留合法边界

写一个测试函数，检查下面三个输入都触发 ValueError：

- `decode([False], id_to_token)`：False 同样是布尔值，不是本课允许的编号。
- `decode(None, id_to_token)`：编号参数应是列表。
- `encode(True, token_to_id)`：原文参数应是字符串。

再检查两个合法边界：`decode([0], ...)` 得到“我”，`decode([6], ...)` 得到“<unk>”。一共新增 5 项检查。

每次异常检查都重新设置标志；仅捕获预期的 ValueError。防止把“拒绝非法输入”实现成“把合法输入也全部拒绝”。


In [ ]:
exercise_c_bad_ids = [[False], None]
# 在这里写异常检查，以及编号 0、6 的正常输出检查。
# 定义测试函数后主动调用；全部通过后再打印成功信息。


---

## 参考答案

下方三个答案包含各自依赖，可单独复制运行。先完成自己的实现，再对照思路。

### A：根据词表写固定预期


In [20]:
def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


token_to_id = {
    "我": 0, "爱": 1, "学": 2, "习": 3,
    "模": 4, "型": 5, "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]


def test_more_encode_cases(token_to_id):
    cases = [
        {"text": "模型", "expected": [4, 5]},
        {"text": "火星", "expected": [6, 6]},
        {"text": "我 爱", "expected": [0, 6, 1]},
    ]
    for case in cases:
        actual = encode(case["text"], token_to_id)
        expected = case["expected"]
        message = "输入：" + repr(case["text"])
        message = message + "，预期：" + repr(expected) + "，实际：" + repr(actual)
        assert actual == expected, message
    return len(cases)

print("新增编码检查通过：", test_more_encode_cases(token_to_id), "项")


新增编码检查通过： 3 项


### B：每遇到一个未知字符，就累加一次

故障版的 count = 1 会反复把计数设成 1，所以无法正确统计两个未知字符。修复为 count = count + 1；计数初值仍在循环外。


In [21]:
def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


token_to_id = {
    "我": 0, "爱": 1, "学": 2, "习": 3,
    "模": 4, "型": 5, "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]


def count_unknown_buggy(text, token_to_id):
    token_ids = encode(text, token_to_id)
    count = 0
    for token_id in token_ids:
        if token_id == token_to_id["<unk>"]:
            count = 1  # 故意写错：请根据测试定位。
    return count


try:
    actual = count_unknown_buggy("我爱大大模型", token_to_id)
    assert actual == 2, "预期 2，实际 " + repr(actual)
except AssertionError as error:
    print("已复现计数故障：", error)


def count_unknown(text, token_to_id):
    token_ids = encode(text, token_to_id)
    count = 0
    for token_id in token_ids:
        if token_id == token_to_id["<unk>"]:
            count = count + 1
    return count


count_cases = [
    {"text": "我爱学习", "expected": 0},
    {"text": "我爱大模型", "expected": 1},
    {"text": "我爱大大模型", "expected": 2},
    {"text": "", "expected": 0},
    {"text": "大大", "expected": 2},
]
for case in count_cases:
    actual = count_unknown(case["text"], token_to_id)
    assert actual == case["expected"], "计数不符合预期：" + repr(case["text"])
print("计数修复后的回归检查通过：", len(count_cases), "项")


已复现计数故障： 预期 2，实际 1
计数修复后的回归检查通过： 5 项


### C：分别检查应当拒绝和应当接受的输入

两类检查需要同时保留。只证明“有异常”不够，还要确认异常类型符合约定；正常边界也应仍能解码。


In [22]:
def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored


token_to_id = {
    "我": 0, "爱": 1, "学": 2, "习": 3,
    "模": 4, "型": 5, "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]


def test_additional_boundaries(token_to_id, id_to_token):
    bad_ids_cases = [[False], None]
    for bad_ids in bad_ids_cases:
        raised_expected_error = False
        try:
            decode(bad_ids, id_to_token)
        except ValueError:
            raised_expected_error = True
        assert raised_expected_error, "decode 应拒绝：" + repr(bad_ids)

    raised_expected_error = False
    try:
        encode(True, token_to_id)
    except ValueError:
        raised_expected_error = True
    assert raised_expected_error, "encode 应拒绝布尔值原文"

    assert decode([0], id_to_token) == "我", "编号 0 应正常解码"
    assert decode([6], id_to_token) == "<unk>", "编号 6 应正常解码"
    return 5

print("新增异常与合法边界检查通过：", test_additional_boundaries(token_to_id, id_to_token), "项")


新增异常与合法边界检查通过： 5 项


## 完成本课的标准

- 能给函数写出输入、固定预期和 assert 检查，并主动调用测试。
- 能区分普通输入、空输入、重复输入与非法输入的预期行为。
- 能解释为什么单字符检查通过，仍可能漏掉 return 缩进错误。
- 能用两个字符或空输入复现问题，打印关键状态，再修复缩进。
- 能让“没有触发预期异常”的错误行为导致测试失败。
- 能在修复后保留原预期，重跑已有检查，解释有限案例的边界。

建议提交练习 A 的测试函数、练习 B 的修复前后结果，以及练习 C 的 5 项检查。最终脚本通过是示例验证，独立完成这些变化题才是学习验收。

**下一课：第 08 课，向量、点积与形状直觉。** 用列表和循环实现点积，用手算结果与本课的检查方法验证计算，开始进入大模型所需的数学基础。

## 官方参考资料

- [Python 3.12：assert 与 return 语句](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html)
- [Python 3.12：错误与异常](https://docs.python.org/zh-cn/3.12/tutorial/errors.html)
- [Python 3.12：repr 内置函数](https://docs.python.org/zh-cn/3.12/library/functions.html#repr)
- [Python 3.12：__debug__ 常量](https://docs.python.org/zh-cn/3.12/library/constants.html#__debug__)
- [Python 3.12：traceback.format_exc](https://docs.python.org/zh-cn/3.12/library/traceback.html#traceback.format_exc)
